In [1]:
import os
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [5]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [8]:
DATA_DIR = "./data2"
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

TRAIN_FILE = os.path.join(
    PROCESSED_DIR,
    "train.csv"
)

VAL_FILE = os.path.join(
    PROCESSED_DIR,
    "val.csv"
)

TEST_FILE = os.path.join(
    PROCESSED_DIR,
    "test.csv"
)

VOCAB_FILE = os.path.join(
    PROCESSED_DIR,
    "vocab.json"
)

## Ucitavanje vokabulara i podataka

In [9]:
with open(VOCAB_FILE, "r", encoding="utf-8") as f:
    vocab_data = json.load(f)

token_to_id = vocab_data["token_to_id"]
id_to_token = {
    int(k): v
    for k, v in vocab_data["id_to_token"].items()
}

MAX_LENGTH = vocab_data["max_length"]

VOCAB_SIZE = len(token_to_id)

PAD_ID = token_to_id["<PAD>"]
UNK_ID = token_to_id["<UNK>"]
SOS_ID = token_to_id["<SOS>"]
EOS_ID = token_to_id["<EOS>"]

In [10]:
train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)
test_df = pd.read_csv(TEST_FILE)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 32360
Validation: 4045
Test: 4050


In [11]:
# Zato sto ucitavanje CSV fajlova liste vidi kao stringove 
def parse_list(value):
    return json.loads(value.replace("'", '"'))

train_df["input_ids"] = train_df["input_ids"].apply(parse_list)
val_df["input_ids"] = val_df["input_ids"].apply(parse_list)
test_df["input_ids"] = test_df["input_ids"].apply(parse_list)

train_df["attention_mask"] = train_df["attention_mask"].apply(parse_list)
val_df["attention_mask"] = val_df["attention_mask"].apply(parse_list)
test_df["attention_mask"] = test_df["attention_mask"].apply(parse_list)

In [42]:
print(train_df.iloc[0]["input_ids"])
print(train_df.iloc[0]["attention_mask"])

[2, 60, 1275, 3398, 60, 4905, 2068, 3480, 1367, 7275, 60, 5833, 4454, 6430, 3398, 216, 2273, 7487, 13, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## PyTorch Dataset

In [16]:
class FlickrTextDataset(Dataset):

    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        input_ids = torch.tensor(
            row["input_ids"],
            dtype=torch.long
        )

        attention_mask = torch.tensor(
            row["attention_mask"],
            dtype=torch.long
        )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "caption": row["caption"],
            "image": row["image"]
        }

In [17]:
train_dataset = FlickrTextDataset(train_df)
val_dataset = FlickrTextDataset(val_df)
test_dataset = FlickrTextDataset(test_df)

print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Test examples:", len(test_dataset))

Train examples: 32360
Validation examples: 4045
Test examples: 4050


In [46]:
sample = train_dataset[42]

print("Caption:")
print(sample["caption"])

print("\nInput IDs:")
print(sample["input_ids"])

print("\nAttention mask:")
print(sample["attention_mask"])

Caption:
a young boy runs aross the street .

Input IDs:
tensor([   2,   60, 7724,  850, 5639,  311, 6895, 6557,   13,    3,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0])

Attention mask:
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])


## Data Loader

In [47]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [48]:
batch = next(iter(train_loader))

print("Input IDs shape:")
print(batch["input_ids"].shape)

print("\nAttention mask shape:")
print(batch["attention_mask"].shape)

Input IDs shape:
torch.Size([64, 32])

Attention mask shape:
torch.Size([64, 32])


## Positional Encoding
Svakom tokenu dodajemo informaciju o njegovoj poziciji u rečenici.

$$
PE(pos, 2i) =
\sin\left(
\frac{pos}{10000^{2i/d_{\text{model}}}}
\right)
$$

$$
PE(pos, 2i+1) =
\cos\left(
\frac{pos}{10000^{2i/d_{\text{model}}}}
\right)
$$

Gde su:


- **$pos$** — pozicija tokena u sekvenci.
- **$i$** — indeks sinusno-kosinusnog para dimenzija.
- **$d_{\text{model}}$** — dimenzija embedding vektora.
- **$2i$** — parna dimenzija, u kojoj koristimo sinus.
- **$2i+1$** — neparna dimenzija, u kojoj koristimo kosinus.
- **$10000$** — određuje raspon frekvencija; različite dimenzije koriste različito brze sinusne talase.

Intuitivno, svaka pozicija dobija svoj jedinstveni vektor pomoću više sinusnih i kosinusnih talasa različitih frekvencija. Taj vektor se zatim dodaje token embeddingu.


In [64]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_length):
        super().__init__()

        position = torch.arange(max_length).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-np.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_length, d_model)

        pe[:, 0::2] = torch.sin(position * div_term)

        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## 1. Transformer

In [68]:
class MiniTextTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_length,
        embedding_dim=256,
        num_heads=4,
        num_layers=2,
        feedforward_dim=512,
        output_dim=256,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_ID
        )

        self.position_encoding = PositionalEncoding(
            d_model=embedding_dim,
            max_length=max_length
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.projection = nn.Linear(
            embedding_dim,
            output_dim
        )

    def forward(self, input_ids, attention_mask):

        # [B, L]
        x = self.embedding(input_ids)

        # [B, L, D]
        x = self.position_encoding(x)

        padding_mask = attention_mask == 0

        x = self.transformer(
            x,
            src_key_padding_mask=padding_mask
        )

        # [B, L, D] -> [B, D]
        mask = attention_mask.unsqueeze(-1).float()

        x = x * mask

        x = x.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

        # [B, D] -> [B, output_dim]
        x = self.projection(x)

        x = F.normalize(x, dim=-1)

        return x

In [65]:
model = MiniTextTransformer(
    vocab_size=VOCAB_SIZE,
    max_length=MAX_LENGTH,

    embedding_dim=256,
    num_heads=4,
    num_layers=2,

    feedforward_dim=512,

    output_dim=256,

    dropout=0.1
).to(DEVICE)

print(model)

MiniTextTransformer(
  (embedding): Embedding(7747, 256, padding_idx=0)
  (position_encoding): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (projection): Linear(in_features=256, out_features=256, bias=True)
)


In [66]:
num_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Trainable parameters: {num_parameters:,}"
)

Trainable parameters: 3,103,232


In [67]:
caption_1 = test_dataset[0]
caption_2 = test_dataset[1]

ids_1 = caption_1["input_ids"].unsqueeze(0).to(DEVICE)
mask_1 = caption_1["attention_mask"].unsqueeze(0).to(DEVICE)

ids_2 = caption_2["input_ids"].unsqueeze(0).to(DEVICE)
mask_2 = caption_2["attention_mask"].unsqueeze(0).to(DEVICE)

with torch.no_grad():
    emb_1 = model(ids_1, mask_1)
    emb_2 = model(ids_2, mask_2)

similarity = F.cosine_similarity(
    emb_1,
    emb_2
)

print("Caption 1:")
print(caption_1["caption"])

print("\nCaption 2:")
print(caption_2["caption"])

print("\nCosine similarity:", similarity.item())

Caption 1:
a couple and an infant , being held by the male , sitting next to a pond with a near by stroller .

Caption 2:
a couple sit on the grass with a baby and stroller .

Cosine similarity: 0.862162709236145


## 2. RNN

In [3]:

class TextEncoderRNN(nn.Module):
    def __init__(self, vocab_size, max_length, embedding_dim=256, hidden_dim=256, num_layers=1, output_dim=256, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.rnn = nn.GRU( embedding_dim,
                           hidden_dim, num_layers=num_layers,
                           batch_first=True,
                           bidirectional=True,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.projection = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        lengths = attention_mask.sum(dim=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        h = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        out = self.projection(h)
        return F.normalize(out, dim=-1)

In [12]:
model = TextEncoderRNN(
    vocab_size=VOCAB_SIZE,
    max_length=MAX_LENGTH,
    embedding_dim=256,
    hidden_dim=256,
    num_layers=1,
    output_dim=256,
    dropout=0.1
).to(DEVICE)

print(model)

num_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_parameters:,}")

TextEncoderRNN(
  (embedding): Embedding(7747, 256, padding_idx=0)
  (rnn): GRU(256, 256, batch_first=True, bidirectional=True)
  (projection): Linear(in_features=512, out_features=256, bias=True)
)
Trainable parameters: 2,904,064


In [13]:
def encode_caption(dataset_item, model):
    ids = dataset_item["input_ids"].unsqueeze(0).to(DEVICE)
    mask = dataset_item["attention_mask"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return model(ids, mask)

def compare_captions(idx_1, idx_2, dataset, model):
    item_1 = dataset[idx_1]
    item_2 = dataset[idx_2]

    emb_1 = encode_caption(item_1, model)
    emb_2 = encode_caption(item_2, model)

    similarity = F.cosine_similarity(emb_1, emb_2).item()

    print("Caption 1:")
    print(item_1["caption"])
    print("\nCaption 2:")
    print(item_2["caption"])
    print("\nCosine similarity:", similarity)
    print("-" * 60)

In [21]:
# Slični parovi (isti pojmovi/scena) — očekuje se veća sličnost nakon treninga
compare_captions(0, 1, test_dataset, model)

# Nasumičan, verovatno nepovezan par
import random
i, j = random.sample(range(len(test_dataset)), 2)
compare_captions(i, j, test_dataset, model)

Caption 1:
a couple and an infant , being held by the male , sitting next to a pond with a near by stroller .

Caption 2:
a couple sit on the grass with a baby and stroller .

Cosine similarity: 0.9732707738876343
------------------------------------------------------------
Caption 1:
a woman wearing headphones walks down the street .

Caption 2:
several people are wading in a river in a deep gorge .

Cosine similarity: 0.7809757590293884
------------------------------------------------------------


## 3. DIstilBERT

In [23]:
from transformers import AutoModel
from image_encoder import build_projection_head  # isti pattern kao za image encoder

class TextEncoderPretrained(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", emb_dim=256, freeze_backbone=True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.freeze_backbone = freeze_backbone
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        hidden_size = self.backbone.config.hidden_size  # 768 za distilbert-base
        self.projection = build_projection_head(hidden_size, emb_dim)

    def forward(self, input_ids, attention_mask):
        if self.freeze_backbone:
            with torch.no_grad():
                out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        else:
            out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)

        # masked mean pooling preko tokena (isti princip kao u MiniTextTransformer)
        token_embeddings = out.last_hidden_state          # [B, L, H]
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

        x = self.projection(pooled)
        return F.normalize(x, dim=-1)

In [24]:
class ContrastiveFlickrDatasetBERT(Dataset):
    def __init__(self, df, images_dir, tokenizer, max_length=32, transform=None, random_caption=True):
        self.df = df
        self.images_dir = images_dir
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.transform = transform
        self.random_caption = random_caption
        self.image_names = self.df["image"].unique()
        self.captions_by_image = {
            name: group.reset_index(drop=True)
            for name, group in self.df.groupby("image")
        }

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        captions = self.captions_by_image[image_name]
        row_index = random.randrange(len(captions)) if self.random_caption else 0
        row = captions.iloc[row_index]

        image = Image.open(os.path.join(self.images_dir, image_name)).convert("RGB")
        if self.transform:
            image = self.transform(image)

        encoded = self.tokenizer(
            row["caption"], padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )

        return {
            "image": image,
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "caption": row["caption"],
            "image_name": image_name
        }